# 02 · Trim and clean the 311 data
**Input:** the full 311 data in `data/raw/311_full/`
**Outputs:**
- `data/interim/311_study_subset.parquet`: the trimmed raw subset
- `data/processed/complaints_clean.parquet`: the cleaned data
- `reports/cleaning_report_311.json`: the audit trail

Every step is a documented function in `src/preprocess.py`. This notebook runs them one at a time and shows what each one changed.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from src import config, transform as T, preprocess as P

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 60)
print("Types:", config.COMPLAINT_TYPES)
print("Window:", config.START_DATE, "→", config.END_DATE, "| columns:", len(config.COLUMNS_311))

## 1. Trim: extract the study subset from the full data
Only months inside the window are opened, only the 17 columns are read, and the type filter is an exact match.

In [ ]:
files = T.month_files()
subset, extract_log = T.extract_study_subset(files)
path = T.save_subset(subset)
print(f"Kept {len(subset):,} of {extract_log.rows_in_month.sum():,} requests in the window "
      f"({len(subset) / extract_log.rows_in_month.sum():.1%}) → {path.name} ({path.stat().st_size / 1e6:.0f} MB)")
extract_log.set_index("month").T

In [ ]:
# Cross-check against the earlier direct API download of the same scope
direct = T.trimmed_download_rows()
print(f"From full data: {len(subset):,} | direct download: {direct:,} | difference: {len(subset) - direct:+,}")
print("(Small differences are normal: Open Data updates records daily.)")
subset["complaint_type"].value_counts().to_frame("requests")

## 2. Clean step by step

In [ ]:
raw = subset            # keep the raw subset for before/after comparisons
rep = P.CleaningReport("311")
rep.add("load_subset", len(raw), len(raw), f"{raw['complaint_type'].nunique()} complaint types")
df = P.drop_duplicate_requests(raw.copy(), rep)
df = P.parse_dates(df, rep)

### Text standardization: placeholders and casing

In [ ]:
print("Borough values BEFORE:"); display(raw["borough"].value_counts(dropna=False).to_frame().T)
df = P.standardize_text(df, rep)
print("Borough values AFTER:");  display(df["borough"].value_counts(dropna=False).to_frame().T)

### Locations: ZIP, BBL, coordinates and borough

In [ ]:
bad_zip = raw["incident_zip"].dropna()
print("Examples of ZIP values that fail validation:")
display(bad_zip[~bad_zip.str.fullmatch(r"1(0[0-4]\d\d|1[0-6]\d\d)")].value_counts().head(8).to_frame().T)
print("BBL examples before:", raw["bbl"].dropna().sample(5, random_state=1).tolist())
df = P.clean_locations(df, rep)

### Closure time: compute it and flag suspicious values
Flagged rows are **kept**. Each analysis decides whether to exclude them, which avoids silently deleting data.

In [ ]:
df = P.add_closure_features(df, rep)
df = P.add_time_features(df)
flags = [c for c in df.columns if c.startswith("flag_")]
display(df[flags].sum().rename("rows flagged").to_frame().T)
df.groupby("complaint_type", observed=True)["closure_hours"].median().rename("median closure hours").to_frame().T

## 3. Audit trail and save

In [ ]:
steps = pd.DataFrame(rep.steps)[["step", "rows_before", "rows_after", "rows_removed", "detail"]]
display(steps)

config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
config.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
df = df.sort_values("created_date").reset_index(drop=True)
df.to_parquet(config.CLEAN_311_FILE, index=False)
(config.REPORTS_DIR / "cleaning_report_311.json").write_text(json.dumps(rep.to_dict(), indent=2, default=str))
print(f"Saved {len(df):,} rows × {df.shape[1]} columns → {config.CLEAN_311_FILE.relative_to(ROOT)}")

In [ ]:
# Final check: missing share of the key fields after cleaning
df[["incident_zip", "bbl", "borough", "latitude", "closed_date", "closure_hours"]].isna().mean().mul(100).round(2) \
    .rename("% missing").to_frame().T

Missing ZIPs are filled from PLUTO via the BBL in notebook 05, once PLUTO is clean.

→ Next: **03_explore_weather_pluto.ipynb**